In [1]:
import numpy as np
import pandas as pd

# Load the dataset 
from sklearn import datasets
iris = datasets.load_iris()

# DataFrame
data = pd.DataFrame(iris.data, columns=iris.feature_names)
data['label'] = iris.target

# Shuffle
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Split 
split_ratio = 0.7
split_index = int(len(data) * split_ratio)
train_data, test_data = data[:split_index], data[split_index:]

# Convert 
X_train, y_train = train_data.iloc[:, :-1].values, train_data.iloc[:, -1].values
X_test, y_test = test_data.iloc[:, :-1].values, test_data.iloc[:, -1].values


In [2]:
import math

# entropy
def entropy(y):
    unique_classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities))

# information gain
def information_gain(y, y_left, y_right):
    p_left, p_right = len(y_left) / len(y), len(y_right) / len(y)
    return entropy(y) - (p_left * entropy(y_left) + p_right * entropy(y_right))


In [3]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label  


In [4]:
def build_tree(X, y, depth=0, max_depth=5):
    # Stopping condition: If pure class or max depth reached
    if len(set(y)) == 1 or depth >= max_depth:
        return Node(label=max(set(y), key=list(y).count))  # Majority class

    best_gain = 0
    best_feature, best_threshold = None, None
    best_left_X, best_right_X, best_left_y, best_right_y = None, None, None, None

    # Iterate to find best split
    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            left_mask, right_mask = X[:, feature] <= threshold, X[:, feature] > threshold
            if sum(left_mask) == 0 or sum(right_mask) == 0:
                continue  #empty splits

            y_left, y_right = y[left_mask], y[right_mask]
            gain = information_gain(y, y_left, y_right)

            if gain > best_gain:
                best_gain = gain
                best_feature, best_threshold = feature, threshold
                best_left_X, best_right_X = X[left_mask], X[right_mask]
                best_left_y, best_right_y = y_left, y_right

    # If no good split found, return leaf
    if best_gain == 0:
        return Node(label=max(set(y), key=list(y).count))

    # Recursion
    left_subtree = build_tree(best_left_X, best_left_y, depth + 1, max_depth)
    right_subtree = build_tree(best_right_X, best_right_y, depth + 1, max_depth)
    
    return Node(feature=best_feature, threshold=best_threshold, left=left_subtree, right=right_subtree)


In [5]:
def prune_tree(node, X_val, y_val):
    if node.left and node.right:
        prune_tree(node.left, X_val, y_val)
        prune_tree(node.right, X_val, y_val)

        # Check if making it a leaf improves accuracy
        if node.left.label is not None and node.right.label is not None:
            leaf_label = max(set(y_val), key=list(y_val).count)  # Majority class
            original_accuracy = np.mean([predict(node, x) == y for x, y in zip(X_val, y_val)])
            node.label = leaf_label
            node.left, node.right = None, None
            pruned_accuracy = np.mean([predict(node, x) == y for x, y in zip(X_val, y_val)])

            # If pruning reduces accuracy, revert
            if pruned_accuracy < original_accuracy:
                node.label = None
                node.left, node.right = left_subtree, right_subtree


In [8]:
# Train the decision tree model
tree = build_tree(X_train, y_train, max_depth=5)


In [9]:
def predict(node, x):
    if node.label is not None:
        return node.label  # Leaf node

    if x[node.feature] <= node.threshold:
        return predict(node.left, x)
    else:
        return predict(node.right, x)

y_pred = np.array([predict(tree, x) for x in X_test])
accuracy = np.mean(y_pred == y_test)
error_rate = 1 - accuracy

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Error Rate: {error_rate * 100:.2f}%")


Accuracy: 95.56%
Error Rate: 4.44%


In [10]:
def print_tree(node, depth=0):
    if node.label is not None:
        print("  " * depth + f"Leaf: Class {node.label}")
    else:
        print("  " * depth + f"[Feature {node.feature}] <= {node.threshold}")
        print_tree(node.left, depth + 1)
        print_tree(node.right, depth + 1)

print_tree(tree)


[Feature 2] <= 1.9
  Leaf: Class 0
  [Feature 2] <= 4.8
    [Feature 3] <= 1.6
      Leaf: Class 1
      [Feature 0] <= 5.9
        Leaf: Class 1
        Leaf: Class 2
    [Feature 3] <= 1.7
      [Feature 3] <= 1.5
        [Feature 2] <= 4.9
          Leaf: Class 1
          Leaf: Class 2
        Leaf: Class 1
      Leaf: Class 2


In [12]:
import matplotlib.pyplot as plt
from graphviz import Digraph

def visualize_tree(node, graph=None, parent=None, edge_label=""):
    if graph is None:
        graph = Digraph()
    
    if node is None:
        return
    
    node_label = f"Feature {node.feature}\nThreshold: {node.threshold}"
    node_id = str(id(node))
    
    graph.node(node_id, node_label)
    
    if parent:
        graph.edge(parent, node_id, label=edge_label)
    
    visualize_tree(node.left, graph, node_id, "Left")
    visualize_tree(node.right, graph, node_id, "Right")
    
    return graph

# Generate the decision tree visualization
graph = visualize_tree(tree)
graph.render("decision_tree", format="png", cleanup=False)  # Save as PNG


'decision_tree.png'